# 1. Setup Environment

In [ ]:
!git clone https://github.com/clovaai/deep-text-recognition-benchmark
%cd deep-text-recognition-benchmark
!pip install lmdb pillow torchvision nltk natsort torch

# 2. Upload and Extract Dataset

In [ ]:
from google.colab import files

print("Please upload easyocr_lmdb_dataset_ready.zip...")
uploaded = files.upload()

if 'easyocr_lmdb_dataset_ready.zip' not in uploaded:
    print("ERROR: Please upload easyocr_lmdb_dataset_ready.zip first!")
else:
    print("Upload complete! Extracting...")
    !unzip -q -o easyocr_lmdb_dataset_ready.zip
    !mkdir -p dataset
    !mv train_lmdb dataset/ 2>/dev/null || true
    !mv val_lmdb dataset/ 2>/dev/null || true
    print("Extracted and moved to dataset/!")

# 3. Download original weights, Patch bug, and Fine-Tune

In [ ]:
import subprocess
import sys
import os
import urllib.request
import zipfile

# 1. Download the actual pre-trained english_g2 model
if not os.path.exists("english_g2.pth"):
    print("Downloading pretrained english_g2 model...")
    urllib.request.urlretrieve("https://github.com/JaidedAI/EasyOCR/releases/download/v1.3/english_g2.zip", "english_g2.zip")
    with zipfile.ZipFile("english_g2.zip", 'r') as zip_ref:
        zip_ref.extractall(".")
    print("Extracted english_g2.pth!")

# 2. Patch train.py so it doesn't overwrite our custom characters and works on CPU/GPU
print("Patching train.py...")
with open("train.py", "r", encoding="utf-8") as f:
    code = f.read()
# Disable the hardcoded overwrite
code = code.replace("opt.character = string.printable[:-6]", "pass # opt.character OVERWRITE DISABLED")
# Fix CUDA deserialization issue if running on CPU
code = code.replace("torch.load(opt.saved_model)", "torch.load(opt.saved_model, map_location=device)")
with open("train.py", "w", encoding="utf-8") as f:
    f.write(code)

# 3. Patch dataset.py to fix PyTorch _accumulate import error!
print("Patching dataset.py...")
with open("dataset.py", "r", encoding="utf-8") as f:
    code = f.read()
code = code.replace("from torch._utils import _accumulate", "from itertools import accumulate as _accumulate")
with open("dataset.py", "w", encoding="utf-8") as f:
    f.write(code)

# 3. The EXACT character string used by JaidedAI (including the degree symbol °)
chars = "0123456789!\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~ °ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"

cmd = [
    "python", "train.py",
    "--train_data", "dataset/train_lmdb",
    "--valid_data", "dataset/val_lmdb",
    "--select_data", "/",
    "--batch_ratio", "1",
    "--Transformation", "None",
    "--FeatureExtraction", "VGG",
    "--SequenceModeling", "BiLSTM",
    "--Prediction", "CTC",
    "--saved_model", "english_g2.pth", # Use the fully pretrained model
    "--FT",                            # Fine-tune mode (keeps prediction weights!)
    "--output_channel", "256",         # Original model dimensions
    "--hidden_size", "256",            # Original model dimensions
    "--batch_size", "128",
    "--data_filtering_off",
    "--workers", "4",
    "--num_iter", "5000",              # Only need 5000 steps since we're fine-tuning
    "--lr", "0.1",                     # Default (1.0) is tuned for from-scratch training on
                                        # millions of samples; kept low so gradients aren't
                                        # violent relative to this dataset's size.
    "--imgW", "512",                   # Wide enough that CTC's ~127 timesteps comfortably
                                        # exceed the longest word-level label (max 40 chars).
    "--PAD",                           # Resize proportionally to imgH then right-pad to imgW,
                                        # instead of squashing to a fixed box. Matches how
                                        # EasyOCR resizes crops at inference time.
    "--valInterval", "100",            # Finer-grained checkpoints so the actual best iteration
                                        # can be pinpointed instead of only sampling every 500.
    "--exp_name", "apex_wordlevel_finetune",
    "--sensitive",
    "--batch_max_length", "256",
    "--character", chars
]

# NOTE: this dataset (easyocr_lmdb_wordlevel_ready, uploaded as easyocr_lmdb_dataset_ready.zip)
# is built differently from the first two training attempts:
#   1. Word-level crops, not whole killfeed lines. ocr_with_easyocr() at inference recognizes
#      each CRAFT-detected word box separately -- it never sees a whole line, and never emits
#      "<GUN_ICON>" itself (that marker is inserted afterward by a Python-side pixel-gap
#      heuristic). Training on whole lines labeled with the literal "<GUN_ICON>" text taught
#      the model a task it's never actually asked to perform at inference. These crops are
#      individual CRAFT-detected word boxes with the marker text removed from labels entirely.
#   2. Correct color polarity. The previous LMDB-build script copied raw crop files directly
#      (light background, dark text). Inference always runs preprocess_for_easyocr() first,
#      which inverts colors (dark background, light text) before OCR. These crops are taken
#      from the *preprocessed* image, so training now sees the same polarity as inference.
# Both were confirmed via direct pixel inspection, not assumption -- see prepare_wordlevel_dataset.py.

print("Starting fine-tuning! Output will stream below...")
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    sys.stdout.write(line)
    sys.stdout.flush()
process.wait()


# 4. Download Trained Weights

In [ ]:
from google.colab import files
files.download('saved_models/apex_wordlevel_finetune/best_accuracy.pth')